# L15 capstone: a pretraining run that survives its own failures

Choose **Runtime, Change runtime type, T4 GPU**, then run the cells in order. You will write the
parts of a real training script: a learning-rate schedule that depends only on the step,
checkpoints that capture everything needed to resume exactly, deterministic data positions, a
spike detector with rollback, and an evaluation harness with error bars. Then you run the
nanoGPT-sized character model (6 layers, width 384, about 10.7 million parameters) on Tiny
Shakespeare for 5,000 steps, with a stretch of corrupted data injected at step 3,000, and write
the run report. The last cell is the autograder; it prints a completion code to paste into the
lesson.

Checkpoints go to Google Drive if it is mounted (`from google.colab import drive;
drive.mount('/content/drive')`), so a disconnected session can resume where it stopped; otherwise
to a temporary folder.

In [ ]:
import hashlib
import json
import math
import os
import pathlib
import tempfile
import time
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SMALL = os.environ.get("SMALL") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

## 1. Configuration

One dictionary holds every setting; it is saved in every checkpoint and in the report. The data
positions in `BAD` will be replaced by random characters, a corrupted shard.

In [ ]:
CFG = dict(d=384, n_layer=6, n_head=6, ctx=256, batch=32, dropout=0.2, steps=5000, lr=1e-3, warmup=200,
           weight_decay=0.1, clip=1.0, eval_every=250, ckpt_every=250, eval_windows=200,
           bad_start=3000, bad_len=10, window=50, k=6.0, skip_ahead=20, seed=0)
if SMALL:
    CFG.update(d=32, n_layer=2, n_head=2, ctx=32, batch=8, dropout=0.0, steps=120, lr=3e-3, warmup=10,
               eval_every=20, ckpt_every=20, eval_windows=40, bad_start=80, bad_len=3, window=20)
BAD = set(range(CFG["bad_start"], CFG["bad_start"] + CFG["bad_len"]))
drive = pathlib.Path("/content/drive/MyDrive")
CKPT_DIR = pathlib.Path(os.environ.get("FORGE_CKPT_DIR") or (drive / "forge-L15" if drive.is_dir() and not SMALL else tempfile.mkdtemp(prefix="forge-L15-")))
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("checkpoints in", CKPT_DIR)

## 2. Data at fixed positions

Batch number `pos` is drawn from a generator seeded with `(seed, pos)`, so any batch can be
regenerated, and skipping data is just moving `pos`. Validation windows are fixed once.

In [ ]:
CORPUS_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
CORPUS_SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
local = [p for p in (pathlib.Path("data/tinyshakespeare.txt"), pathlib.Path("../data/tinyshakespeare.txt")) if p.exists()]
raw = local[0].read_bytes() if local else urllib.request.urlopen(CORPUS_URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == CORPUS_SHA256, "unexpected corpus contents"
text = raw.decode("utf-8")
chars = sorted(set(text))
VOCAB = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]


def get_batch(pos, cfg=CFG, source=None):
    source = train_data if source is None else source
    g = torch.Generator().manual_seed(cfg["seed"] * 1_000_003 + pos)
    ix = torch.randint(len(source) - cfg["ctx"] - 1, (cfg["batch"],), generator=g)
    x = torch.stack([source[i:i + cfg["ctx"]] for i in ix])
    y = torch.stack([source[i + 1:i + cfg["ctx"] + 1] for i in ix])
    if pos in BAD and source is train_data:
        x = torch.randint(VOCAB, x.shape, generator=g)
        y = torch.randint(VOCAB, y.shape, generator=g)
    return x.to(DEVICE), y.to(DEVICE)


_vg = torch.Generator().manual_seed(12345)
VAL_STARTS = torch.randint(len(val_data) - CFG["ctx"] - 1, (CFG["eval_windows"],), generator=_vg)

## 3. The model

The L10 GPT with dropout, tied embeddings and PyTorch's fused attention.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_head, dropout):
        super().__init__()
        self.n_head, self.dropout = n_head, dropout
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(dropout))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(d, dim=2)
        q, k, v = (t.view(B, T, self.n_head, d // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        x = x + self.drop(self.proj(y.transpose(1, 2).reshape(B, T, d)))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok, self.pos = nn.Embedding(VOCAB, cfg["d"]), nn.Embedding(cfg["ctx"], cfg["d"])
        nn.init.normal_(self.tok.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([Block(cfg["d"], cfg["n_head"], cfg["dropout"]) for _ in range(cfg["n_layer"])])
        self.ln_f = nn.LayerNorm(cfg["d"])

    def forward(self, idx):
        x = self.drop(self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device)))
        for b in self.blocks:
            x = b(x)
        return self.ln_f(x) @ self.tok.weight.T

## 4. The schedule

Write `lr_at(step, cfg)`: linear warmup to `cfg["lr"]` over `cfg["warmup"]` steps, then cosine
decay to 10% of the peak at `cfg["steps"]`, as a pure function of the step (L15 exercise 5).

In [ ]:
def lr_at(step, cfg=CFG):
    raise NotImplementedError("Write your code here")

## 5. Checkpoints

Write `save_checkpoint(path, run)` and `load_checkpoint(path, run)`. `run` is a dictionary with
the model, optimizer, gradient scaler, `step`, `pos` (the next data position), `skips` and the
metrics `log`. A checkpoint must hold everything needed to continue exactly: the state dicts, the
counters, the log, the configuration and the random-number state of the CPU (and of the GPU, if
there is one), because dropout draws from it. Write to a temporary file and rename it into place,
so a crash during saving never leaves a half-written checkpoint.

In [ ]:
def save_checkpoint(path, run):
    raise NotImplementedError("Write your code here")


def load_checkpoint(path, run):
    raise NotImplementedError("Write your code here")


def new_run(cfg=CFG):
    torch.manual_seed(cfg["seed"])
    model = GPT(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], betas=(0.9, 0.95), weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    return {"model": model, "opt": opt, "scaler": scaler, "step": 0, "pos": 0, "skips": [], "log": [], "cfg": cfg}


def train_step(run):
    """One optimizer step on the batch at run["pos"]; returns the loss and the gradient norm before
    clipping. The step count and the data position both advance."""
    cfg, model, opt, scaler = run["cfg"], run["model"], run["opt"], run["scaler"]
    x, y = get_batch(run["pos"], cfg)
    run["step"] += 1
    run["pos"] += 1
    for group in opt.param_groups:
        group["lr"] = lr_at(run["step"], cfg)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
        loss = F.cross_entropy(model(x).flatten(0, 1).float(), y.flatten())
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"]).item()
    scaler.step(opt)
    scaler.update()
    return loss.item(), norm

### Exact resume, checked on a tiny model

Ten steps, a checkpoint, ten more steps must give bitwise the same weights as twenty steps
straight. This check runs on the CPU, where every operation is deterministic.

In [ ]:
def exact_resume_check():
    global DEVICE
    device, DEVICE = DEVICE, "cpu"
    tiny = dict(CFG, d=32, n_layer=2, n_head=2, ctx=16, batch=4, dropout=0.1, steps=20, warmup=5)
    try:
        a = new_run(tiny)
        for _ in range(20):
            train_step(a)
        b = new_run(tiny)
        for _ in range(10):
            train_step(b)
        path = CKPT_DIR / "resume-check.pt"
        save_checkpoint(path, b)
        for _ in range(3):
            train_step(b)  # work after the checkpoint, lost in the "crash"
        c = load_checkpoint(path, new_run(tiny))
        for _ in range(10):
            train_step(c)
        return all(torch.equal(p, q) for p, q in zip(a["model"].parameters(), c["model"].parameters())) and c["step"] == 20
    finally:
        DEVICE = device


resume_exact = exact_resume_check()
print("exact resume:", resume_exact)

## 6. Evaluation with error bars

Write `evaluate(model)`: the mean cross-entropy over the fixed validation windows `VAL_STARTS`,
in evaluation mode and without gradients, with a 95% percentile bootstrap interval over windows
(2,000 resamples, generator seeded with 0). Return `(mean, low, high, per_window_losses)`.

In [ ]:
@torch.no_grad()
def evaluate(model, cfg=CFG):
    raise NotImplementedError("Write your code here")

## 7. The run

The loop checks each step's loss **before** trusting the update: when it exceeds the mean of the
previous `window` kept losses by `k` standard deviations, it restores the last checkpoint saved
before the previous step, moves the data position `skip_ahead` batches past the flagged one, and
continues (L15 exercises 4 and 9). Write the spike check `is_spike(history, loss, cfg)`; the rest
of the loop is given. It resumes from the latest checkpoint in `CKPT_DIR` if there is one.

In [ ]:
def is_spike(history, loss, cfg=CFG):
    raise NotImplementedError("Write your code here")


def checkpoint_path(step):
    return CKPT_DIR / f"step-{step:06d}.pt"


run = new_run()
existing = sorted(CKPT_DIR.glob("step-*.pt"))
if existing:
    load_checkpoint(existing[-1], run)
    print("resumed from", existing[-1].name)
else:
    save_checkpoint(checkpoint_path(0), run)
history = [e["loss"] for e in run["log"] if "loss" in e]
spikes = [e for e in run["log"] if e.get("event") == "rollback"]
t0, tokens = time.time(), 0
while run["step"] < CFG["steps"]:
    pos = run["pos"]
    loss, norm = train_step(run)
    tokens += CFG["batch"] * CFG["ctx"]
    if is_spike(history, loss):
        back = (run["step"] - 2) // CFG["ckpt_every"] * CFG["ckpt_every"]
        load_checkpoint(checkpoint_path(back), run)
        event = {"event": "rollback", "flagged_pos": pos, "restored_step": back, "skipped": [run["pos"], pos + CFG["skip_ahead"]]}
        run["log"].append(event)
        run["skips"].append((run["pos"], pos + CFG["skip_ahead"]))
        run["pos"] = pos + CFG["skip_ahead"] + 1
        history = [e["loss"] for e in run["log"] if "loss" in e]
        spikes.append(event)
        print(f"spike at data position {pos} (loss {loss:.2f}); rolled back to step {back}, data now at {run['pos']}")
        continue
    history.append(loss)
    run["log"].append({"step": run["step"], "pos": pos, "loss": loss, "grad_norm": norm, "lr": lr_at(run["step"]),
                       "tokens_per_s": tokens / (time.time() - t0)})
    if run["step"] % CFG["eval_every"] == 0 or run["step"] == CFG["steps"]:
        mean, lo, hi, _ = evaluate(run["model"])
        run["log"].append({"step": run["step"], "event": "eval", "val": mean, "val_low": lo, "val_high": hi})
        print(f"step {run['step']:5d}: train {loss:.3f}, val {mean:.3f} [{lo:.3f}, {hi:.3f}], grad norm {norm:.2f}, "
              f"lr {lr_at(run['step']):.2e}, {tokens / (time.time() - t0):,.0f} tokens/s")
    if run["step"] % CFG["ckpt_every"] == 0:
        save_checkpoint(checkpoint_path(run["step"]), run)

evals = [e for e in run["log"] if e.get("event") == "eval"]
final_val, final_low, final_high, final_per = evaluate(run["model"])
before = [e for e in evals if e["step"] < CFG["bad_start"]]
pre_spike_val = before[-1]["val"] if before else float("nan")
with open(CKPT_DIR / "metrics.json", "w") as f:
    json.dump(run["log"], f)
print(f"final validation loss {final_val:.4f} [{final_low:.4f}, {final_high:.4f}]; {len(spikes)} rollback(s)")

## 8. Report

In `REPORT`, write at least 80 words: the configuration, the final validation loss with its
interval, what happened at the spike (when it was detected, where the run rolled back to and
which data it skipped), and what you would monitor or change in a longer run.

In [ ]:
REPORT = """Write your report here."""
print(REPORT)

In [ ]:
# Forge autograder: run this cell last. It runs the capstone tests and prints your completion code.
import hashlib as _hashlib, sys as _sys, types as _types, unittest as _unittest
_FORGE_ID = "L15"
_FORGE_TESTS = '''import math
import torch
import sub


def test_schedule():
    cfg = dict(lr=1e-3, warmup=100, steps=1000)
    assert sub.lr_at(1, cfg) == 1e-5 and sub.lr_at(100, cfg) == 1e-3
    assert abs(sub.lr_at(1000, cfg) - 1e-4) < 1e-15 and abs(sub.lr_at(550, cfg) - 5.5e-4) < 1e-15


def test_exact_resume():
    assert sub.resume_exact is True


def test_checkpoints():
    files = sorted(sub.CKPT_DIR.glob("step-*.pt"))
    assert files and not list(sub.CKPT_DIR.glob("*.tmp"))
    state = torch.load(files[-1], map_location="cpu", weights_only=False)
    for key in ["model", "opt", "scaler", "step", "pos", "log", "cfg", "rng_cpu"]:
        assert key in state, key


def test_spike_detected_and_rolled_back():
    cfg = sub.CFG
    assert 1 <= len(sub.spikes) <= 2, sub.spikes
    s = sub.spikes[0]
    assert cfg["bad_start"] <= s["flagged_pos"] <= cfg["bad_start"] + 2, s
    assert s["restored_step"] < cfg["bad_start"] and s["restored_step"] % cfg["ckpt_every"] == 0
    assert s["skipped"][1] >= cfg["bad_start"] + cfg["bad_len"] - 1


def test_log():
    steps = [e for e in sub.run["log"] if "loss" in e]
    assert [e["step"] for e in steps] == list(range(1, sub.CFG["steps"] + 1))
    assert all(math.isfinite(e["loss"]) and e["grad_norm"] >= 0 and e["lr"] > 0 and e["tokens_per_s"] > 0 for e in steps)
    assert all(e["pos"] not in sub.BAD for e in steps)


def test_evaluation_and_recovery():
    assert sub.final_low <= sub.final_val <= sub.final_high and len(sub.final_per) == sub.CFG["eval_windows"]
    first = sub.evals[0]["val"]
    assert sub.final_val < first
    if not sub.SMALL:
        assert sub.final_val < 1.70 and sub.final_val < sub.pre_spike_val, (sub.final_val, sub.pre_spike_val)


def test_report():
    text = sub.REPORT.lower()
    assert len(text.split()) >= 80
    assert "spike" in text and "validation" in text and "rolled back" in text
'''
sub = _types.SimpleNamespace(**{_k: _v for _k, _v in globals().items() if not _k.startswith('_')})
_sys.modules['sub'] = sub
_ns = {'__name__': 'forge_tests'}
exec(compile(_FORGE_TESTS, 'tests.py', 'exec'), _ns)
_passed = []
for _name, _fn in list(_ns.items()):
    if _name.startswith('test_') and callable(_fn):
        try:
            _fn()
            _passed.append(_name)
            print('PASS', _name)
        except _unittest.SkipTest as _e:
            print('SKIP', _name, '-', _e)
        except Exception as _e:
            print('FAIL', _name, '-', type(_e).__name__, _e)
_digest = _hashlib.sha256(f"{_FORGE_ID}:{','.join(sorted(_passed))}:forge".encode()).hexdigest()[:8]
print(f"Completion code: FORGE-{_FORGE_ID}-{_digest}")